# Frequency-domain FWI with `odil_wave.experiment`


## Imports & repo paths


In [ ]:
# Force inline (interactive) figures so plt.show() renders in the notebook.
%matplotlib inline

import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

import odil_wave
from odil_wave.experiment import (
    resolve_config, build_problem, run_inverse, new_run_directory,
    LiveVelocityView, postprocess_run,
    load_tape, load_loss_tape, load_final_velocity,
)
from odil_wave.metrics import ssim, ssim_map

ODIL_ROOT = Path(odil_wave.__file__).resolve().parents[1]
print("odil root      :", ODIL_ROOT)

## 1. Choose a configuration

In [ ]:
OVERDENSITY_BASE = 1500.0
OVERDENSITY_CONTRAST = 400.0
OVERDENSITY_RADIUS = 0.05

# Run config: only the values that differ from the default config
# (`python main.py --print-defaults` prints it).
# Short run (~2 min on CPU): a 32-transducer ring and a two-band 40 -> 60 kHz
# continuation, 20 joint L-BFGS outer steps per band.
RUN_CONFIG = {
    "grid": {"space_order": 4},
    "init": {"profile": "homogeneous"},
    "acquisition": {"n_transducers": 32},
    "continuation": {
        "bands": [
            {"frequencies_hz": [40e3]},
            {"frequencies_hz": [60e3]},
        ]
    },
}

user_cfg = RUN_CONFIG
cfg = resolve_config(user_cfg)
print("truth        :", cfg.truth.profile,
      f"| base {cfg.truth.base:.0f} + contrast {cfg.truth.contrast:.0f} m/s"
      f" | radius {cfg.truth.extra.get('radius')} m")
print("grid interior:", cfg.grid.interior_shape, "| pml:", cfg.pml.width)
print("observation  : helmholtz on truth | pml:", cfg.pml.observation_width)
print("bands [kHz]  :", [[f / 1e3 for f in b.frequencies_hz] for b in cfg.continuation.bands])

## 2. Build the problem and reproduce the setup diagnostics

`build_problem(cfg)` builds the grid, source, acquisition layout and truth /
initial models once — the same object every solver operates on. Per-band pieces
(frequency selection, geometry, observed wavefields) come from
`problem.make_band(...)`.


In [ ]:
problem = build_problem(cfg)
grid = problem.grid
print(grid.summary)

band0 = problem.make_band(cfg.continuation.bands[0].frequencies_hz)
print("selected frequencies [kHz]:", (band0.freq.frequencies / 1e3).tolist())
print("FFT bins                  :", band0.freq.fft_bins.tolist())

grid.plot_absorption_profile(); plt.show()
band0.geom.show(problem.truth_velocity)
band0.geom.show(problem.init_velocity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
problem.source.plot_pulse(ax=axes[0])
band0.geom.plot_source_field(src_idx=0, ax=axes[1])
plt.tight_layout(); plt.show()

# Source magnitude spectrum with the inversion bins marked (library helper).
problem.source.plot_spectrum(frequencies=band0.freq)
plt.show()

In [ ]:
# Observed (truth) wavefield on the dominant bin. Wavefield.show now draws a
# 2x2 of |u|, Re(u), Im(u) and phase in one call.
k = int(np.argmin(np.abs(band0.freq.frequencies.abs().cpu().numpy() - problem.source.f0)))
band0.observed_wfs[0].show(k, title="Observed shot 0 (truth)")

## 3. Run the inversion — one call, one artifact directory

`run_inverse` builds the problem, runs every continuation band with the selected
optimiser, logs the metric tape, and saves each band's velocity field into
`outputs/runs/<YYMMDD-HHMMSS>_<sha7>[_scratch]/`. A notebook usually runs on
a dirty tree, so this is a **scratch run** (`git.diff` saved, never cited).


In [ ]:
live = LiveVelocityView()
result = run_inverse(cfg, new_run_directory(scratch=True), on_log=live)
run_dir = result.run_dir

print("status  :", result.status)
print("run_dir :", run_dir)
print("summary :", json.dumps(result.metrics_summary, indent=2))

## 4. The per-iteration metric tape

`tape.csv` holds one row per logged iterate (see the run-layer README). Every
run ends with post-processing, which draws the standard figures from the run
directory alone into `<run_dir>/figures/`; `convergence.png` plots the tape
against `iter_total`. `load_loss_tape` rebuilds a `LossTape` so its own
`show()` works offline.

In [ ]:
records = load_tape(run_dir)
print(f"{len(records)} tape rows; columns:")
print(list(records[0].keys()))
display(Image(filename=str(run_dir / "figures" / "convergence.png")))

# The saved scalars also reconstruct a LossTape, so its built-in plots work
# fully offline.
tape = load_loss_tape(run_dir)
tape.show(title="Loss (reconstructed from disk)")

## 5. Velocity recovery — from the saved artifacts

`velocity.png` is truth | recovered | recovered − truth with brain and skull
outlines; with more than one band, `per_band_error.png` shows each band's
error and `band_contribution.png` / `.csv` what each band changed at what cost.
`profiles.png` / `.csv` cut through the head centre. `ssim_map` shows the
per-pixel SSIM inside the head ROI.

In [ ]:
figures = run_dir / "figures"
for name in ["velocity", "per_band_error", "profiles", "band_contribution"]:
    if (figures / f"{name}.png").exists():  # per-band figures need > 1 band
        display(Image(filename=str(figures / f"{name}.png")))

# Per-pixel SSIM map inside the head ROI, recomputed from disk.
c_final_t = load_final_velocity(run_dir).c.to(problem.truth_velocity.c)
s = ssim(c_final_t, problem.truth_velocity.c, grid, mask=problem.head_mask)
print(f"final head-ROI SSIM (recomputed from disk): {s:.4f}")
ssim_map(c_final_t, problem.truth_velocity.c, grid,
         title=f"SSIM map (SSIM={s:.3f})", mask=problem.head_mask)
plt.show()

In [ ]:
# Redraw the figures of any saved run (same as `main.py --postprocess <run_dir>`);
# post-processing reads only the run directory and never re-runs a solve.
# run_plot_dir = ODIL_ROOT / "outputs" / "runs" / "<YYMMDD-HHMMSS>_<sha7>_scratch"
# postprocess_run(run_plot_dir)

## 6. Dig deeper: on-demand plots

One-line library calls on a run directory (no CLI, not saved). Like
post-processing they read only the run directory and never solve or optimise;
the PDE residual map evaluates the operator on the saved `u` and `c`. `band`
defaults to the last completed band, `shot` to its worst-fit shot, `frequency`
to the band's first solved frequency.


In [ ]:
from odil_wave.experiment import on_demand

display(on_demand.source_spectrum(run_dir))   # inversion bins on the source spectrum
display(on_demand.optimiser_health(run_dir))  # c extremes vs bounds, sat_frac, gradient norms
display(on_demand.wavefield_comparison(run_dir, band=-1, shot=None, frequency=0))  # u | u* | u - u*
display(on_demand.shot_data_fit(run_dir, shot=None))  # d_obs vs d_syn against receiver angle
display(on_demand.pde_residual_map(run_dir, shot=None))  # |r_pde| / (|λ_tt|·u_ref), full grid
